# Preparación del manifiesto OAI para landing

Permite inspeccionar las dos salidas de `oai_load_identifiers` sin escribir en PostgreSQL. Las funciones deben mantenerse alineadas con `src/kedro_cic/pipelines/oai_load/nodes.py`. Ejecutar desde `kedro jupyter lab`.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [ ]:
df_identifiers_raw = catalog.load("raw/oai/identifiers#parquet")
df_identifiers_raw.head()

In [ ]:
def _pick_load_datetime(df: pd.DataFrame) -> pd.Timestamp:
    """Return the batch load timestamp, preserving an existing value if present."""
    for column in ("load_datetime", "_load_datetime"):
        if column not in df.columns:
            continue
        values = pd.to_datetime(df[column], errors="coerce", utc=True).dropna()
        if not values.empty:
            return values.max()
    return pd.Timestamp.now(tz="UTC")

In [ ]:
def _normalize_extract_datetime(df: pd.DataFrame) -> pd.DataFrame:
    """Expose the raw extraction timestamp under the landing contract name."""
    if "extract_datetime" in df.columns:
        return df
    if "_extract_datetime" not in df.columns:
        raise ValueError("OAI records require _extract_datetime")
    return df.rename(columns={"_extract_datetime": "extract_datetime"})

In [ ]:
def oai_load_identifiers(df_identifiers_raw: pd.DataFrame) -> pd.DataFrame:
    df_identifiers_raw = _normalize_extract_datetime(df_identifiers_raw.copy())
    load_dt = _pick_load_datetime(df_identifiers_raw)
    identifier_columns = ["record_id", "datestamp", "is_deleted", "extract_datetime", "_context", "_source_key", "_repository_identifier", "_institution_ror", "_base_url", "_metadata_prefix"]
    df_identifiers = df_identifiers_raw[identifier_columns].copy()
    df_identifiers_sets = (df_identifiers_raw[["record_id", "set_id", "extract_datetime", "_source_key"]].explode("set_id", ignore_index=True).dropna(subset=["set_id"]))
    df_identifiers["_load_datetime"] = load_dt
    df_identifiers_sets["_load_datetime"] = load_dt
    return df_identifiers, df_identifiers_sets

In [ ]:
df_identifiers, df_identifier_sets = oai_load_identifiers(df_identifiers_raw)
assert df_identifiers["record_id"].notna().all()
assert not df_identifiers["record_id"].duplicated().any()
assert df_identifier_sets["record_id"].notna().all()
assert df_identifiers["_source_key"].notna().all()
{"identifiers": len(df_identifiers), "deleted": int(df_identifiers["is_deleted"].fillna(False).sum()), "identifier_sets": len(df_identifier_sets), "sources": df_identifiers["_source_key"].value_counts().to_dict()}

In [ ]:
display(df_identifiers.head())
display(df_identifier_sets.head())